# Stats tools from zero (tutorial before P11)

Run every cell top to bottom and read the output. Do the **Your turn** cells yourself. Allow about 30 min.

**The whole topic in one idea:**
You only see a **sample** (say 500 customers, not all 26 million). Any number you work out from a sample **wobbles**: a different sample would give a slightly different answer.
- The **SE (standard error)** measures how much it wobbles.
- A **CI (confidence interval)** is: `your estimate ± critical value (a number from a table) × SE (standard error)`.
- A **hypothesis test** asks how many SEs (standard errors) away from "no difference" your result is. Far away means it's unlikely to be luck.

Everything below is just those three lines, worked out with real numbers.

### Glossary (every symbol is also spelled out where it's used)

| Symbol | Say it out loud | Meaning |
|---|---|---|
| `n` | **n (sample size / group size)** | how many values or people |
| `x` | **x (the data)**, or **x (number who said yes)** for rates | |
| `m`, mean | **mean (average)** | |
| `s`, std | **s (sample standard deviation)** | typical distance of a value from the mean (average) |
| `ddof` | **ddof (delta degrees of freedom)** | `ddof=1` → divide by n − 1 |
| SE | **SE (standard error)** | how much an estimate wobbles between samples |
| CI | **CI (confidence interval)** | estimate ± critical value × SE (standard error) |
| `conf` | **conf (confidence level)** | e.g. 0.95 for 95% |
| `q` | **q (quantile: fraction of the curve below a point)** | e.g. 0.975 |
| `crit` | **crit (critical value)** | the "number from a table", e.g. 1.96 |
| `df` | **df (degrees of freedom)** | n − 1 for a t-distribution CI |
| `p`, `p_a` | **p (proportion / rate)** | e.g. 45 / 900 = 0.05. **Not** the p-value! |
| p-value | **p-value** | how surprising the data is if H0 (null hypothesis) were true |
| H0 | **H0 (null hypothesis)** | "there is no difference" |
| `z` | **z (z-score)** | how many SEs (standard errors) from zero |
| `cdf` | **cdf (cumulative distribution function)** | fraction of the curve **below** a point |
| `ppf` | **ppf (percent point function)** | the reverse of cdf: point with a given fraction below |
| `sf` | **sf (survival function)** | fraction of the curve **above** a point |
| `sqrt` | **sqrt (square root)** | |

## 1. numpy basics: mean (average), std (standard deviation), sqrt (square root)

Six ATM (cash machine) withdrawals, in £. One customer's week.

In [1]:
import numpy as np
from scipy import stats

x = np.array([20, 40, 60, 20, 100, 60])   # x (the data)

print("n (sample size)                =", len(x))           # how many values
print("mean (average)                 =", x.mean())         # (20+40+60+20+100+60) / 6
print("s (sample standard deviation)  =", x.std(ddof=1))    # typical distance from the mean
print("sqrt (square root) of 9        =", np.sqrt(9))

n (sample size)                = 6
mean (average)                 = 50.0
s (sample standard deviation)  = 30.331501776206203
sqrt (square root) of 9        = 3.0


**What's `ddof=1` (delta degrees of freedom = 1)?** numpy's default `x.std()` (standard deviation) divides by `n` (sample size). For a **sample** you divide by `n − 1` instead, which is slightly bigger and corrects for the sample's mean (average) sitting a bit too close to its own data.
`ddof=1` means "divide by n − 1". **For sample data, always use `ddof=1`.**

In [2]:
print("std (standard deviation), divide by n     (ddof=0):", x.std(ddof=0))
print("std (standard deviation), divide by n - 1 (ddof=1):", x.std(ddof=1))   # <- the one you want

std (standard deviation), divide by n     (ddof=0): 27.688746209726915
std (standard deviation), divide by n - 1 (ddof=1): 30.331501776206203


**Your turn 1.** For `y` below, set `n` (sample size), `m` (mean) and `s` (sample standard deviation).

In [3]:
y = np.array([5.0, 7.0, 9.0, 11.0])
n = len(y)
m = y.mean()
s = y.std(ddof=1)

assert n == 4 and m == 8.0 and np.isclose(s, 2.5820, atol=1e-4)
print("correct")

correct


## 2. SE (standard error): how much the mean (average) wobbles

**`SE (standard error) of the mean = s (sample standard deviation) / sqrt(n) (square root of the sample size)`**

Don't take it on trust: below we make a fake "population" of 100,000 spend amounts, take 2,000 random samples of 100 customers, and work out each sample's mean (average). The spread of those 2,000 means should match `s / sqrt(100)`.

In [4]:
rng = np.random.default_rng(2)                          # rng (random number generator)
population = rng.lognormal(3.3, 0.5, size=100_000)     # skewed, like real spend

sample_means = [rng.choice(population, 100).mean() for _ in range(2000)]
print("spread of 2000 sample means (averages):                    ", round(np.std(sample_means), 2))

one_sample = rng.choice(population, 100)
print("SE (standard error) = s / sqrt(n) worked out from ONE sample:", round(one_sample.std(ddof=1) / np.sqrt(100), 2))

spread of 2000 sample means (averages):                     1.63
SE (standard error) = s / sqrt(n) worked out from ONE sample: 1.57


The two numbers are close. So **from one sample** you can estimate how much your mean (average) would wobble. That's the trick behind all of this.
- Bigger `n` (sample size) → smaller SE (standard error): more data, less wobble. Because of the sqrt (square root), **4× the data only halves the SE (standard error)**.

Our ATM example: `SE (standard error) = 30.33 / sqrt(6)`.

In [5]:
se = x.std(ddof=1) / np.sqrt(len(x))   # se (standard error) = s (sample standard deviation) / sqrt(n)
print("SE (standard error) =", se)

SE (standard error) = 12.382783747337808


## 3. scipy.stats: reading the normal (bell) curve

The normal (bell) curve tells you how far from the middle things usually land. You only need **3 functions**. `stats.norm` is the standard bell curve: centre 0, width 1.

| Function | Question it answers | Example |
|---|---|---|
| `stats.norm.cdf(z)`: **cdf (cumulative distribution function)** | What fraction is **below** z (z-score)? | `cdf(1.96)` → 0.975 |
| `stats.norm.ppf(q)`: **ppf (percent point function)** | What z (z-score) has fraction q **below** it? (the reverse of cdf) | `ppf(0.975)` → 1.96 |
| `stats.norm.sf(z)`: **sf (survival function)** | What fraction is **above** z (z-score)? (= 1 − cdf) | `sf(1.96)` → 0.025 |

Use **ppf (percent point function)** to **build a CI (confidence interval)**: it gives the critical value.
Use **sf (survival function)** to **get a p-value**: how rare is my z (z-score).

In [6]:
print("cdf (cumulative distribution function) at 1.96 =", stats.norm.cdf(1.96))    # 97.5% of the curve is below 1.96
print("ppf (percent point function) at 0.975          =", stats.norm.ppf(0.975))   # reverse: which z has 97.5% below it?
print("sf (survival function) at 1.96                 =", stats.norm.sf(1.96))     # 2.5% is above 1.96

cdf (cumulative distribution function) at 1.96 = 0.9750021048517795
ppf (percent point function) at 0.975          = 1.959963984540054
sf (survival function) at 1.96                 = 0.024997895148220435


**Why 0.975 for a 95% CI (confidence interval)?** 95% in the middle leaves 5% outside, split as **2.5% in each tail**. The right-hand cut-off has 97.5% below it.

```
        2.5%  |          95%           |  2.5%
   -----------|------------------------|-----------
           -1.96          0           +1.96
```

General formula: `q (fraction below) = 1 - (1 - conf (confidence level)) / 2`. For conf = 0.95 that's 0.975; for 0.99 it's 0.995.

In [7]:
for conf in (0.90, 0.95, 0.99):                  # conf (confidence level)
    q = 1 - (1 - conf) / 2                       # q (fraction of the curve below the cut-off)
    print(f"conf (confidence level) {conf}: q = {q}, crit (critical value) = {stats.norm.ppf(q):.3f}")

conf (confidence level) 0.9: q = 0.95, crit (critical value) = 1.645
conf (confidence level) 0.95: q = 0.975, crit (critical value) = 1.960
conf (confidence level) 0.99: q = 0.995, crit (critical value) = 2.576


**Your turn 2.** Set `crit` (critical value) for an **80%** CI (confidence interval), using the formula, not a hard-coded q (fraction below).

In [8]:
conf = 0.80
crit = stats.norm.ppf(1 - (1 - conf) / 2)     # q = 0.90 -> 1.2816
assert np.isclose(crit, 1.2816, atol=1e-4)
print("correct")

correct


## 4. t-distribution instead of normal (small samples, means only)

When you build a CI (confidence interval) for a **mean (average)**, you used `s` (sample standard deviation) from the sample, and `s` is itself a guess. With few data points that guess is shaky, so use the **t-distribution**: same idea, fatter tails, so a bigger critical value and a wider CI (confidence interval).

`stats.t.ppf(q, df)`: ppf (percent point function) of the t-distribution, with **df (degrees of freedom) = n (sample size) − 1**. As n (sample size) grows, t turns into the normal.

In [9]:
for n_ in (5, 30, 1001):    # n_ (sample size)
    print(f"n (sample size) = {n_:>4}: t crit (critical value) = {stats.t.ppf(0.975, n_ - 1):.3f}   (normal = 1.960)")

n (sample size) =    5: t crit (critical value) = 2.776   (normal = 1.960)
n (sample size) =   30: t crit (critical value) = 2.045   (normal = 1.960)
n (sample size) = 1001: t crit (critical value) = 1.962   (normal = 1.960)


**Rule:**
- CI (confidence interval) for a **mean (average)** → `stats.t.ppf(q, n - 1)`, the t-distribution with df (degrees of freedom) = n − 1.
- CI (confidence interval) or test for a **proportion (rate**, like 5% applied) → `stats.norm.ppf(q)`, the normal.

## 5. Worked example: 95% CI (confidence interval) for a mean (average), every step

Back to the 6 ATM withdrawals. Read each line, then run it.

In [10]:
x = np.array([20, 40, 60, 20, 100, 60])      # x (the data)
conf = 0.95                                  # conf (confidence level)

n      = len(x)                              # n (sample size)                 = 6
mean   = x.mean()                            # mean (average)                  = 50.0
s      = x.std(ddof=1)                       # s (sample standard deviation)   = 30.33
se     = s / np.sqrt(n)                      # se (standard error)             = 12.38
q      = 1 - (1 - conf) / 2                  # q (fraction below the cut-off)  = 0.975
t_crit = stats.t.ppf(q, n - 1)               # t crit (critical value), df (degrees of freedom) = 5 -> 2.571
lo, hi = mean - t_crit * se, mean + t_crit * se   # lo / hi (lower / upper end of the CI)

print(f"mean (average) = {mean}, 95% CI (confidence interval) = ({lo:.2f}, {hi:.2f})")

mean (average) = 50.0, 95% CI (confidence interval) = (18.17, 81.83)


**In words:** "Average withdrawal £50, 95% CI (confidence interval) £18 to £82." Wide, because n (sample size) is only 6. With 600 values it would be much tighter.

Check against scipy's ready-made version (you're **not** allowed to use this in P11, but it proves the steps are right):

In [11]:
print(stats.t.interval(0.95, n - 1, loc=mean, scale=se))   # loc (centre) = mean, scale = se (standard error)

(np.float64(18.169041024480855), np.float64(81.83095897551914))


## 6. Proportions (rates)

A **proportion (rate)** = how many said yes / how many there were. Example: a fraud alert went to n = 900 customers (group size) and **x = 45** (number who said yes) clicked "this wasn't me".

`p (proportion) = 45 / 900 = 0.05`, and the **SE (standard error) of a proportion = `sqrt(p × (1 − p) / n)`**. You don't need a std (standard deviation) here: for yes/no data, `p(1 − p)` *is* the variance (std squared).

In [12]:
x_yes, n = 45, 900                           # x_yes (number who said yes), n (group size)
p  = x_yes / n                               # p (proportion / rate)          = 0.05
se = np.sqrt(p * (1 - p) / n)                # se (standard error) of a rate  = 0.00726
z_crit = stats.norm.ppf(0.975)               # z crit (critical value) = 1.96 (proportion -> normal, not t)
print(f"rate = {p:.1%}, 95% CI (confidence interval) = ({p - z_crit*se:.2%}, {p + z_crit*se:.2%})")

rate = 5.0%, 95% CI (confidence interval) = (3.58%, 6.42%)


## 7. Worked example: comparing two groups (the A/B test)

An **A/B test** = show two versions to two random groups and compare. An email campaign:
- **A** (old email): n_a = 600 sent (group size), x_a = **30** clicked (number who said yes).
- **B** (new email): n_b = 600 sent (group size), x_b = **45** clicked (number who said yes).

Is B really better, or is it luck?

**Step 1. Rates and difference.**

In [13]:
x_a, n_a = 30, 600       # group A: x (number who clicked), n (group size)
x_b, n_b = 45, 600       # group B: x (number who clicked), n (group size)
p_a, p_b = x_a / n_a, x_b / n_b              # p (proportion / rate) for each group
diff = p_b - p_a                             # diff (difference in rates)
print(f"A rate = {p_a:.1%}, B rate = {p_b:.1%}, difference = {diff:.1%}")

A rate = 5.0%, B rate = 7.5%, difference = 2.5%


**Step 2. The test: how many SEs (standard errors) is that difference from zero?**

H0 (null hypothesis) says "A and B are really the same", so under H0 the best guess of that shared rate uses **both groups together**: the **pooled rate**. Then the SE (standard error) of a difference between two groups:

`SE (standard error) = sqrt(p_pool × (1 − p_pool) × (1/n_a + 1/n_b))`

`z (z-score) = difference / SE (standard error)` → "how many standard errors from zero".

In [14]:
p_pool = (x_a + x_b) / (n_a + n_b)                          # p_pool (pooled proportion) = 75 / 1200 = 0.0625
se_pooled = np.sqrt(p_pool * (1 - p_pool) * (1/n_a + 1/n_b)) # se (standard error), pooled
z = diff / se_pooled                                         # z (z-score): standard errors from zero
print(f"pooled rate = {p_pool}, SE (standard error) = {se_pooled:.5f}, z (z-score) = {z:.2f}")

pooled rate = 0.0625, SE (standard error) = 0.01398, z (z-score) = 1.79


**Step 3. The p-value: how rare is a z (z-score) that big if H0 (null hypothesis) were true?**

"Two-sided" = a difference in **either** direction counts as extreme, so take the area above `|z|` (z-score without its sign) using **sf (survival function)**, **and double it**.

In [15]:
p_value = 2 * stats.norm.sf(abs(z))   # sf (survival function) = area above |z|; x2 for two-sided
print(f"p-value = {p_value:.4f}")

p-value = 0.0736


**Reading it:** p-value ≈ 0.074. *If the two emails were really equally good (H0, the null hypothesis), we'd see a gap this big or bigger about 7% of the time just by luck.* That's above 0.05, so we **can't** call B better yet. Not "B is no better": just "not enough evidence".

**Step 4. CI (confidence interval) for the difference: how big could the real effect be?**

For a CI (confidence interval) we're **not** assuming A = B, so each group keeps its own rate: the **unpooled** SE (standard error).

In [16]:
se_unpooled = np.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)   # se (standard error), unpooled
z_crit = stats.norm.ppf(0.975)                                         # z crit (critical value) = 1.96
lo, hi = diff - z_crit * se_unpooled, diff + z_crit * se_unpooled      # lo / hi (ends of the CI)
print(f"B - A = {diff:.2%}, 95% CI (confidence interval) ({lo:.2%}, {hi:.2%})")

B - A = 2.50%, 95% CI (confidence interval) (-0.24%, 5.24%)


The CI (confidence interval) **includes 0** (from −0.24% to +5.24%), which matches p-value > 0.05. **If a 95% CI (confidence interval) for a difference includes 0, the test at 5% isn't significant**, and the other way round.

**Pooled vs unpooled, one line:** test → assume no difference (H0, null hypothesis) → pool. CI (confidence interval) → measure the difference → don't pool.

**Your turn 3.** Same steps, new numbers: A = 40/500, B = 62/500. Set `z` (z-score) and `p_value`.

In [17]:
x_a, n_a, x_b, n_b = 40, 500, 62, 500
p_a, p_b = x_a / n_a, x_b / n_b
p_pool = (x_a + x_b) / (n_a + n_b)
se_pooled = np.sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))
z = (p_b - p_a) / se_pooled
p_value = 2 * stats.norm.sf(abs(z))

assert np.isclose(z, 2.2987, atol=1e-3) and np.isclose(p_value, 0.0215, atol=1e-3)
print("correct: significant at 5%")

correct: significant at 5%


## Cheat sheet

| I want... | Formula | scipy |
|---|---|---|
| s (sample standard deviation) | `x.std(ddof=1)` | |
| SE (standard error) of a mean (average) | `s / np.sqrt(n)` | |
| SE (standard error) of a proportion (rate) | `np.sqrt(p * (1 - p) / n)` | |
| crit (critical value) for a conf (confidence level) CI | `q = 1 - (1 - conf) / 2` | mean: `stats.t.ppf(q, n - 1)`; proportion: `stats.norm.ppf(q)` |
| CI (confidence interval) | `estimate ± crit * SE` | |
| z (z-score) for two groups (test) | `diff / pooled SE` | |
| two-sided p-value | | `2 * stats.norm.sf(abs(z))`, sf (survival function) |

**Now go back to `11_ab_test_ci.ipynb`.** Each of its 3 functions is one section here wrapped in a `def`:
`mean_ci` = section 5, `two_prop_ztest` = section 7 steps 1-3, `diff_ci` = section 7 step 4.